In [ ]:
from pathlib import Path
import pandas as pd

In [ ]:
PROJECT_ROOT = Path("/home/harielpadillasanchez/Documentos/TT/TT2")

SFT_DIR = PROJECT_ROOT / "data" / "sft_ready"
LORA_DIR = PROJECT_ROOT / "outputs" / "lora_runs"
OUT_DIR = PROJECT_ROOT / "outputs" / "final_sample_36"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TEST_JSONL = SFT_DIR / "feina_repr30_test_sft.jsonl"

LLAMA_PRED = LORA_DIR / "lora_llama3_feina_repr30_v1" / "evaluation" / "lora_test_predictions.csv"
MISTRAL_PRED = LORA_DIR / "lora_mistral_feina_repr30_v1" / "evaluation" / "lora_test_predictions.csv"

print("TEST_JSONL:", TEST_JSONL)
print("LLAMA_PRED:", LLAMA_PRED)
print("MISTRAL_PRED:", MISTRAL_PRED)
print("OUT_DIR:", OUT_DIR)

In [ ]:
test_df = pd.read_json(TEST_JSONL, lines=True)

print("Shape test_df:", test_df.shape)
display(test_df.head(3))
display(test_df.columns.tolist())

In [ ]:
required_cols = ["row_id", "instruction", "output"]
missing = [c for c in required_cols if c not in test_df.columns]
if missing:
    raise ValueError(f"Faltan columnas en test_df: {missing}")

print("row_id únicos:", test_df["row_id"].nunique())

In [ ]:
RANDOM_STATE = 42
N_SAMPLE = 36

sample36_df = (
    test_df
    .sample(n=N_SAMPLE, random_state=RANDOM_STATE)
    .sort_values("row_id")
    .reset_index(drop=True)
)

print("Shape sample36_df:", sample36_df.shape)
display(sample36_df[["row_id", "instruction", "output"]].head(10))

In [ ]:
sample36_jsonl = OUT_DIR / "feina_repr30_test_sample36.jsonl"
sample36_csv = OUT_DIR / "feina_repr30_test_sample36.csv"

sample36_df.to_json(sample36_jsonl, orient="records", lines=True, force_ascii=False)
sample36_df.to_csv(sample36_csv, index=False, encoding="utf-8-sig")

print("Guardado JSONL:", sample36_jsonl)
print("Guardado CSV  :", sample36_csv)

In [ ]:
llama_pred_df = pd.read_csv(LLAMA_PRED)

print("Shape llama_pred_df:", llama_pred_df.shape)
display(llama_pred_df.head(3))
display(llama_pred_df.columns.tolist())

In [ ]:
mistral_pred_df = pd.read_csv(MISTRAL_PRED)

print("Shape mistral_pred_df:", mistral_pred_df.shape)
display(mistral_pred_df.head(3))
display(mistral_pred_df.columns.tolist())

In [ ]:
sample_ids = set(sample36_df["row_id"].tolist())

llama_pred_36 = (
    llama_pred_df[llama_pred_df["row_id"].isin(sample_ids)]
    .copy()
    .sort_values("row_id")
    .reset_index(drop=True)
)

mistral_pred_36 = (
    mistral_pred_df[mistral_pred_df["row_id"].isin(sample_ids)]
    .copy()
    .sort_values("row_id")
    .reset_index(drop=True)
)

print("Shape llama_pred_36  :", llama_pred_36.shape)
print("Shape mistral_pred_36:", mistral_pred_36.shape)

In [ ]:
assert len(llama_pred_36) == 36, f"Llama no quedó en 36, quedó en {len(llama_pred_36)}"
assert len(mistral_pred_36) == 36, f"Mistral no quedó en 36, quedó en {len(mistral_pred_36)}"

assert set(llama_pred_36["row_id"]) == sample_ids
assert set(mistral_pred_36["row_id"]) == sample_ids

print("OK: ambos LoRA quedaron alineados con los mismos 36 row_id")

In [ ]:
llama_pred_36_path = OUT_DIR / "lora_llama3_test_sample36_predictions.csv"
mistral_pred_36_path = OUT_DIR / "lora_mistral_test_sample36_predictions.csv"

llama_pred_36.to_csv(llama_pred_36_path, index=False, encoding="utf-8-sig")
mistral_pred_36.to_csv(mistral_pred_36_path, index=False, encoding="utf-8-sig")

print("Guardado:", llama_pred_36_path)
print("Guardado:", mistral_pred_36_path)

In [ ]:
print("Resumen sample36")
print("-" * 50)
print("Test original:", len(test_df))
print("Submuestra:", len(sample36_df))
print("LoRA llama3:", len(llama_pred_36))
print("LoRA mistral:", len(mistral_pred_36))

print("\nPrimeros row_id sample36:")
print(sample36_df["row_id"].tolist()[:15])